In [ ]:
import scrapy
import pandas as pd
from scrapy.crawler import CrawlerProcess
from scrapy import Request
from scrapy.spiders import Spider
from datetime import datetime
import time

In [ ]:
import os
from mongodb_client import MongoDBClient
mongo_uri = os.getenv("MONGO_URI")
db_name = os.getenv("DB_NAME")
collection = os.getenv("COLLECTION_NAME")

In [ ]:
client = MongoDBClient(mongo_uri, db_name, collection)

In [ ]:
class LosTiemposSpider(Spider):
    name = "lostiempos"
    allowed_domains = ["www.lostiempos.com"]
    start_urls = [
        f"https://www.lostiempos.com/etiqueta/feminicidio?page={i}"for i in range(20, -1,-1)
        
    ]
    
    avoid_sections = ["/tendencias/", "/mundo/", "/tendencias/", "/opinion/", "/politico/","/economia/"]
    

    def __init__(self):
        self.items = []
        self.mongo_client = client
    
    def format_url(self, url):
        new_url = "https://" + self.allowed_domains[0] + url
        return new_url
    
    def date_formatter(self, url, date_format="%Y%m%d"):
        try:
            url_split = url.split("/")
            date_str = url_split[5]
            date_publish = datetime.strptime(date_str, date_format)
            return date_publish
        except Exception as e:
            self.logger.error(f"Error al formatear fecha: {e}")
            return None
    
    def tittle_formatter(self, title):
        try:
            new_title = title.replace("“", '"').replace("”", '"').strip()
            return new_title
        except Exception as e:
            self.logger.error(f"Error al formatear título: {e}")
            return title
    
    def tag_formatter(self, tags):
        try:
            list_tags = [t.lower() for t in tags]
            return list_tags
        except Exception as e:
            self.logger.error(f"Error al formatear tags: {e}")
            return tags
    
    def section_formatter(self, url):
        try:
            url_split = url.split("/")
            section = url_split[4]
            return section
        except Exception as e:
            self.logger.error(f"Error al formatear sección: {e}")
            return url

    def body_formatter(self, body):
        try:
            new_body = [
                b.strip()
                .replace("\xa0", " ")
                .replace('\"', "")
                .replace("\ufeff", " ")
                .replace("“", '"')
                .replace("”", '"')
                .replace("\u200b", " ")
                for b in body
            ]
            new_body = [b for b in new_body if b != " "]
            return new_body
        except Exception as e:
            self.logger.error(f"Error al formatear cuerpo: {e}")
            return body

    def start_requests(self):
        for url in self.start_urls:
            self.logger.info(f"Enviando request a: {url}")
            yield Request(url=url, callback=self.parse_response)
            
    
    def parse_response(self, response):
        self.logger.info(f"Recibida respuesta: {response.url}")
        try:
            noticias = response.xpath('(//div[@class="view-content"])[1]//div[contains(@class, "views-field-title")]/span/a/@href').getall()
            
            self.logger.info(f"Total noticias encontradas: {len(noticias)}")
            for noticia in noticias:
                if not any(s in noticia for s in self.avoid_sections):
                    noticia = self.format_url(noticia)
                    self.logger.info(f"Enviando request a: {noticia}")
                    yield Request(url=noticia, callback=self.parse_news)
        except Exception as e:
            self.logger.error(f"Error al procesar la respuesta JSON: {e}")
            return
    
    def parse_news(self, response):
        title = response.xpath('//h1[@class="node-title"]/text()').get()
        item = {}
        item["url"] = response.url
        item["title"] = self.tittle_formatter(title)
        tags = response.xpath('//ul[@class="field-items"]//li//a/text()').getall()
        item["tags"] = self.tag_formatter(tags)
        item["section"] = self.section_formatter(response.url)
        body = response.xpath('//div[@class="body"]//p/text()').getall()
        item["body"] = self.body_formatter(body)
        item["date_published"] = self.date_formatter(response.url)
        item["source"] = self.name
        self.items.append(item)
        self.logger.info(f"Noticia agregada: {item['url']}")
        time.sleep(2)
    
    def close(self, reason):
        self.mongo_client.connect()
        self.logger.info("Guardando datos en MongoDB")
        for item in self.items:
            try:
                self.mongo_client.insert_new_document(item, "url")
            except Exception as e:
                self.logger.error(f"Error al insertar en MongoDB: {e}")
        self.mongo_client.close()
        self.logger.info(f"Spider cerrado por la razón: {reason}")

In [ ]:
process = CrawlerProcess()
process.crawl(LosTiemposSpider)
process.start()